# Pairwise Fair Comparison for Multivariate Counterfactuals

This notebook reuses the multivariate evaluation setup and compares methods pairwise on the common instances they both evaluated. For non-validity metrics, only shared instances where both methods produced valid counterfactuals are used. NUN validity can be added as an optional extra filter, but it is not required by default.

In [1]:
cd ../..

/home/mrefoyo/Projects/Multi-SpaCE


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf

from IPython.display import display

from experiments.evaluation.evaluation_utils import (
    load_dataset_for_eval,
    calculate_metrics_for_dataset,
    summarize_target_pairwise_comparisons,
)

print(tf.__version__)

2026-05-03 12:18:03.534979: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-03 12:18:03.772367: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 12:18:03.791848: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-05-03 12:18:03.791876: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above

2.10.1


In [29]:
# DATASETS = ['CBF', 'chinatown', 'coffee', 'gunpoint', 'ECG200']
DATASETS = [
    "BasicMotions", 
    "NATOPS", 
    "UWaveGestureLibrary",
    'ArticularyWordRecognition', 
    'Cricket',
    'Epilepsy', 
    'PenDigits', 
    'PEMS-SF', 
    'RacketSports', 'SelfRegulationSCP1'
]
DATASETS = [
    "BasicMotions", 
    "NATOPS", 
]

MO_UTILITY = np.array([0.1, 0.4 * 0.7, 0.6 * 0.7, 0.2])
# MO_UTILITY = np.array([0., 0.4 * 1, 0.6 * 1, 0.])
MODEL_TO_EXPLAIN = 'inceptiontime_noscaling'
SCALING = 'none'
OSC_NAMES = {'AE': 'ae_basic_train', 'IF': 'if_basic_train', 'LOF': 'lof_basic_train'}
METHODS = {
    # 'comte_gpu': 'COMTE',
    # 'abcf_gpu': 'AB-CF',
    # 'discox_gpu': 'DiscoX',
    'mcels': 'M-CELS Sec.',
    "mcels_global": "M-CELS Global",
    "mcels_tensorflow": "M-CELS Sec. TF",
    # 'mascots_scalar_gpu': 'MASCOTS',
    # 'ec57e0a613241455487f3f40483cd34095c81bf7': 'Multi-SpaCE',

    "574aca0df70f6f6d3ceac2fbf11faa9dc1b3b8ce": "Multi-SpaCE Global",
    "74fcb2cbf44799fc5994280f11f3357407396f07": "Multi-SpaCE Sec."
    
}

HIGHER_IS_BETTER_METRICS = {'valid'}
METRICS = ['valid', 'sparsity', 'L2', 'AE_OS', 'IF_OS', 'LOF_OS', 'subsequences', 'subsequences %', '(sparsity + subsequences %) / 2', 'times']
METRIC_NAME_MAP = {
    'valid': 'Validity',
    'sparsity': 'Sparsity',
    'L2': 'Proximity (L2)',
    '(sparsity + subsequences %) / 2': '(Sparsity + Subsequences [%]) / 2',
    'AE_OS': 'AE OS',
    'IF_OS': 'IF OS',
    'LOF_OS': 'LOF OS',
    'subsequences': 'Subsequences',
    'subsequences %': 'Subsequences %',
    'times': 'Time',
}

PENALIZATION_QUANTILES = [0.95]
PENALIZE_INVALID = False

## Get Results

This block is intentionally the same evaluation entry point used in the current multivariate notebook.

In [30]:
data_dict = {}
models_dict = {}
outlier_calculators_dict = {}
possible_nuns_dict = {}
desired_classes_dict = {}
original_classes_dict = {}

mean_results_dict_all = {}
methods_cfs_dict_all = {}
results_all_datasets_df_all = {}
common_test_indexes_dict_all = {}

for dataset in DATASETS:
    print(f'Calculating metrics for {dataset}')
    data_tuple, original_classes, model, outlier_calculators, possible_nuns, desired_classes = load_dataset_for_eval(
        dataset,
        MODEL_TO_EXPLAIN,
        OSC_NAMES,
        scaling=SCALING,
    )
    data_dict[dataset] = data_tuple
    models_dict[dataset] = model
    outlier_calculators_dict[dataset] = outlier_calculators
    possible_nuns_dict[dataset] = possible_nuns
    desired_classes_dict[dataset] = desired_classes
    original_classes_dict[dataset] = original_classes

    dataset_results = calculate_metrics_for_dataset(
        dataset,
        METHODS,
        MODEL_TO_EXPLAIN,
        data_tuple,
        original_classes,
        model,
        outlier_calculators,
        possible_nuns,
        mo_weights=MO_UTILITY,
        penalize_invalid=PENALIZE_INVALID,
        penalization_quantile=PENALIZATION_QUANTILES,
    )

    mean_results_dict_all[dataset] = {key: result['mean_std_df'] for key, result in dataset_results.items()}
    methods_cfs_dict_all[dataset] = {key: result['method_cfs_dataset'] for key, result in dataset_results.items()}
    results_all_datasets_df_all[dataset] = {key: result['results_df'] for key, result in dataset_results.items()}
    common_test_indexes_dict_all[dataset] = {key: result['common_test_indexes'] for key, result in dataset_results.items()}

Calculating metrics for BasicMotions
74fcb2cbf44799fc5994280f11f3357407396f07


100%|██████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:08<00:00,  4.46it/s]


mcels_tensorflow


100%|██████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:00<00:00, 77.83it/s]


mcels


...

100%|████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:12<00:00,  8.18it/s]


mcels_global


100%|████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:11<00:00,  8.57it/s]


574aca0df70f6f6d3ceac2fbf11faa9dc1b3b8ce


100%|████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:18<00:00,  5.53it/s]


In [31]:
selected_penalization_key = 'none'

results_all_datasets_df = pd.concat(
    [results_all_datasets_df_all[dataset][selected_penalization_key] for dataset in DATASETS],
    ignore_index=True,
)

In [32]:
results_all_datasets_df_all[dataset][selected_penalization_key].drop("dataset", axis=1).groupby("method").mean()

,ii,nchanges,sparsity,L1,L2,proba,valid,nuns_valid,AE_OS,IF_OS,LOF_OS,AE_IOS,IF_IOS,LOF_IOS,subsequences,subsequences %,(sparsity + subsequences %) / 2,times,best cf index,order
method,,,,,,,,,,,,,,,,,,,,
M-CELS Global,87.18,50.397849,0.041175,52.303768,7.016293,0.923940,0.93,1.0,0.702330,0.312182,0.228977,0.157135,0.004523,0.006093,6.784946,0.011087,0.026131,26.102248,0.00,4.0
M-CELS Sec.,87.18,53.896907,0.044033,62.844230,8.071375,0.924394,0.97,1.0,0.766685,0.320455,0.257429,0.225879,0.013145,0.025546,8.216495,0.013426,0.028730,23.114070,0.00,3.0
M-CELS Sec. TF,87.18,NaN,NaN,NaN,NaN,NaN,0.00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.831605,0.00,2.0
Multi-SpaCE Global,87.18,133.670000,0.109208,93.021035,8.221725,0.903398,1.00,1.0,0.617711,0.287854,0.205698,0.074694,0.005999,0.006318,9.630000,0.015735,0.062471,9.030909,25.16,5.0
Multi-SpaCE Sec.,87.18,157.150000,0.128391,122.090228,9.878399,0.863379,1.00,1.0,0.635876,0.316801,0.245284,0.091115,0.020258,0.028915,12.830000,0.020964,0.074677,13.474309,23.88,1.0


In [33]:
columns = ["sparsity", "L2", "proba", "valid", "AE_OS", "IF_OS", "LOF_OS", "subsequences", '(sparsity + subsequences %) / 2', "times"]
for dataset in DATASETS:
    print(dataset)
    display(results_all_datasets_df_all[dataset][selected_penalization_key].drop("dataset", axis=1).groupby("method").mean()[columns])

BasicMotions


,sparsity,L2,proba,valid,AE_OS,IF_OS,LOF_OS,subsequences,(sparsity + subsequences %) / 2,times
method,,,,,,,,,,
M-CELS Global,0.098235,52.356071,0.798257,0.85,0.367469,0.289458,0.050737,17.441176,0.078186,14.297759
M-CELS Sec.,0.069048,59.073988,0.867403,0.70,0.492450,0.420300,0.235559,12.535714,0.055417,11.719071
M-CELS Sec. TF,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,9.566741
Multi-SpaCE Global,0.186333,67.666491,0.722080,1.00,0.341911,0.274597,0.042183,4.950000,0.101417,6.283558
Multi-SpaCE Sec.,0.133833,71.569553,0.704624,1.00,0.466997,0.396873,0.189745,7.675000,0.079708,6.306063


NATOPS


,sparsity,L2,proba,valid,AE_OS,IF_OS,LOF_OS,subsequences,(sparsity + subsequences %) / 2,times
method,,,,,,,,,,
M-CELS Global,0.041175,7.016293,0.923940,0.93,0.702330,0.312182,0.228977,6.784946,0.026131,26.102248
M-CELS Sec.,0.044033,8.071375,0.924394,0.97,0.766685,0.320455,0.257429,8.216495,0.028730,23.114070
M-CELS Sec. TF,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,10.831605
Multi-SpaCE Global,0.109208,8.221725,0.903398,1.00,0.617711,0.287854,0.205698,9.630000,0.062471,9.030909
Multi-SpaCE Sec.,0.128391,9.878399,0.863379,1.00,0.635876,0.316801,0.245284,12.830000,0.074677,13.474309


## Fair Pairwise Comparison

For each pair, comparisons are restricted to shared evaluated instances. For non-validity metrics, the compared instances must also be valid for both methods. NUN validity is not enforced unless explicitly requested.

In [25]:
TARGET_METHOD = 'Multi-SpaCE Sec.'

In [26]:
pairwise_summary_df, pairwise_detail_tables = summarize_target_pairwise_comparisons(
    results_df=results_all_datasets_df,
    target_method=TARGET_METHOD,
    metrics=METRICS,
    higher_is_better_metrics=HIGHER_IS_BETTER_METRICS,
    alpha=0.05,
    require_valid_nuns=False,
    require_both_valid_cfs=True,
)

pairwise_summary_df['metric_display'] = pairwise_summary_df['metric'].map(METRIC_NAME_MAP).fillna(pairwise_summary_df['metric'])
pairwise_summary_df['W/T/L'] = (
    pairwise_summary_df['win'].astype(str)
    + '/'
    + pairwise_summary_df['tie'].astype(str)
    + '/'
    + pairwise_summary_df['loss'].astype(str)
)
pairwise_summary_df

,metric,target_method,competitor_method,n_datasets,win,tie,loss,mean_shared_instances,min_shared_instances,max_shared_instances,...,wins_a,wins_b,ties,mean_delta_a_minus_b,median_delta_a_minus_b,better_method,p_value_holm,reject_holm,metric_display,W/T/L
0,(sparsity + subsequences %) / 2,Multi-SpaCE Sec.,M-CELS Global,2,0,0,2,63.5,34.0,93.0,...,0.0,2.0,0.0,-0.023795,-0.023795,M-CELS Global,0.5,False,(Sparsity + Subsequences [%]) / 2,0/0/2
1,(sparsity + subsequences %) / 2,Multi-SpaCE Sec.,M-CELS Sec.,2,0,0,2,62.5,28.0,97.0,...,0.0,2.0,0.0,-0.031659,-0.031659,M-CELS Sec.,0.5,False,(Sparsity + Subsequences [%]) / 2,0/0/2
2,(sparsity + subsequences %) / 2,Multi-SpaCE Sec.,M-CELS Sec. TF,0,0,0,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,(Sparsity + Subsequences [%]) / 2,0/0/0
3,(sparsity + subsequences %) / 2,Multi-SpaCE Sec.,Multi-SpaCE Global,2,1,0,1,70.0,40.0,100.0,...,1.0,1.0,0.0,0.004751,0.004751,Multi-SpaCE Sec.,1.0,False,(Sparsity + Subsequences [%]) / 2,1/0/1
4,AE_OS,Multi-SpaCE Sec.,M-CELS Global,2,1,0,1,63.5,34.0,93.0,...,1.0,1.0,0.0,-0.004199,-0.004199,M-CELS Global,1.0,False,AE OS,1/0/1
5,AE_OS,Multi-SpaCE Sec.,M-CELS Sec.,2,2,0,0,62.5,28.0,97.0,...,2.0,0.0,0.0,0.074678,0.074678,Multi-SpaCE Sec.,0.5,False,AE OS,2/0/0
6,AE_OS,Multi-SpaCE Sec.,M-CELS Sec. TF,0,0,0,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,AE OS,0/0/0
7,AE_OS,Multi-SpaCE Sec.,Multi-SpaCE Global,2,0,0,2,70.0,40.0,100.0,...,0.0,2.0,0.0,-0.071625,-0.071625,Multi-SpaCE Global,0.5,False,AE OS,0/0/2
8,IF_OS,Multi-SpaCE Sec.,M-CELS Global,2,0,0,2,63.5,34.0,93.0,...,0.0,2.0,0.0,-0.039982,-0.039982,M-CELS Global,0.5,False,IF OS,0/0/2
9,IF_OS,Multi-SpaCE Sec.,M-CELS Sec.,2,2,0,0,62.5,28.0,97.0,...,2.0,0.0,0.0,0.004757,0.004757,Multi-SpaCE Sec.,0.5,False,IF OS,2/0/0


In [27]:
for metric in METRICS:
    metric_table = pairwise_summary_df.loc[
        pairwise_summary_df['metric'] == metric,
        [
            'competitor_method',
            'n_datasets',
            'W/T/L',
            'mean_shared_instances',
            'min_shared_instances',
            'max_shared_instances',
            'p_value_raw',
            'p_value_holm',
            'reject_holm',
            'better_method',
        ],
    ].copy()
    if metric_table.empty:
        continue
    metric_table = metric_table.sort_values('competitor_method').reset_index(drop=True)
    print(f"{METRIC_NAME_MAP.get(metric, metric)}: {TARGET_METHOD} vs rest")
    display(metric_table)

Validity: Multi-SpaCE Sec. vs rest


,competitor_method,n_datasets,W/T/L,mean_shared_instances,min_shared_instances,max_shared_instances,p_value_raw,p_value_holm,reject_holm,better_method
0,M-CELS Global,2,2/0/0,70.0,40.0,100.0,0.5,0.5,False,Multi-SpaCE Sec.
1,M-CELS Sec.,2,2/0/0,70.0,40.0,100.0,0.5,0.5,False,Multi-SpaCE Sec.
2,M-CELS Sec. TF,2,2/0/0,70.0,40.0,100.0,0.5,0.5,False,Multi-SpaCE Sec.
3,Multi-SpaCE Global,2,0/2/0,70.0,40.0,100.0,1.0,1.0,False,tie


Sparsity: Multi-SpaCE Sec. vs rest


,competitor_method,n_datasets,W/T/L,mean_shared_instances,min_shared_instances,max_shared_instances,p_value_raw,p_value_holm,reject_holm,better_method
0,M-CELS Global,2,0/0/2,63.5,34.0,93.0,0.5,0.5,False,M-CELS Global
1,M-CELS Sec.,2,0/0/2,62.5,28.0,97.0,0.5,0.5,False,M-CELS Sec.
2,M-CELS Sec. TF,0,0/0/0,NaN,NaN,NaN,NaN,NaN,False,NaN
3,Multi-SpaCE Global,2,1/0/1,70.0,40.0,100.0,1.0,1.0,False,Multi-SpaCE Sec.


Proximity (L2): Multi-SpaCE Sec. vs rest


...

,competitor_method,n_datasets,W/T/L,mean_shared_instances,min_shared_instances,max_shared_instances,p_value_raw,p_value_holm,reject_holm,better_method
0,M-CELS Global,2,1/0/1,63.5,34.0,93.0,1.0,1.0,False,Multi-SpaCE Sec.
1,M-CELS Sec.,2,1/0/1,62.5,28.0,97.0,1.0,1.0,False,Multi-SpaCE Sec.
2,M-CELS Sec. TF,0,0/0/0,NaN,NaN,NaN,NaN,NaN,False,NaN
3,Multi-SpaCE Global,2,0/0/2,70.0,40.0,100.0,0.5,0.5,False,Multi-SpaCE Global


(Sparsity + Subsequences [%]) / 2: Multi-SpaCE Sec. vs rest


,competitor_method,n_datasets,W/T/L,mean_shared_instances,min_shared_instances,max_shared_instances,p_value_raw,p_value_holm,reject_holm,better_method
0,M-CELS Global,2,0/0/2,63.5,34.0,93.0,0.5,0.5,False,M-CELS Global
1,M-CELS Sec.,2,0/0/2,62.5,28.0,97.0,0.5,0.5,False,M-CELS Sec.
2,M-CELS Sec. TF,0,0/0/0,NaN,NaN,NaN,NaN,NaN,False,NaN
3,Multi-SpaCE Global,2,1/0/1,70.0,40.0,100.0,1.0,1.0,False,Multi-SpaCE Sec.


Time: Multi-SpaCE Sec. vs rest


,competitor_method,n_datasets,W/T/L,mean_shared_instances,min_shared_instances,max_shared_instances,p_value_raw,p_value_holm,reject_holm,better_method
0,M-CELS Global,2,2/0/0,63.5,34.0,93.0,0.5,0.5,False,Multi-SpaCE Sec.
1,M-CELS Sec.,2,2/0/0,62.5,28.0,97.0,0.5,0.5,False,Multi-SpaCE Sec.
2,M-CELS Sec. TF,0,0/0/0,NaN,NaN,NaN,NaN,NaN,False,NaN
3,Multi-SpaCE Global,2,0/0/2,70.0,40.0,100.0,0.5,0.5,False,Multi-SpaCE Global


In [28]:
for competitor_method, detail_df in pairwise_detail_tables.items():
    print(f'{TARGET_METHOD} vs {competitor_method}')
    detail_df = detail_df.copy()
    detail_df['metric_display'] = detail_df['metric'].map(METRIC_NAME_MAP).fillna(detail_df['metric'])
    display(
        detail_df[
            [
                'dataset',
                'metric_display',
                'n_common_instances',
                'n_common_valid_nuns',
                'n_metric_instances',
                'method_a_mean',
                'method_b_mean',
                'instance_wins_a',
                'instance_ties',
                'instance_wins_b',
                'dataset_winner',
            ]
        ].sort_values(['metric_display', 'dataset']).reset_index(drop=True)
    )

Multi-SpaCE Sec. vs M-CELS Global


,dataset,metric_display,n_common_instances,n_common_valid_nuns,n_metric_instances,method_a_mean,method_b_mean,instance_wins_a,instance_ties,instance_wins_b,dataset_winner
0,BasicMotions,(Sparsity + Subsequences [%]) / 2,40,40,34,0.079265,0.078186,15,0,19,M-CELS Global
1,NATOPS,(Sparsity + Subsequences [%]) / 2,100,100,93,0.072642,0.026131,18,4,71,M-CELS Global
2,BasicMotions,AE OS,40,40,34,0.444072,0.367469,11,0,23,M-CELS Global
3,NATOPS,AE OS,100,100,93,0.634126,0.702330,66,0,27,Multi-SpaCE Sec.
4,BasicMotions,IF OS,40,40,34,0.363636,0.289458,7,1,26,M-CELS Global
5,NATOPS,IF OS,100,100,93,0.317968,0.312182,40,3,50,M-CELS Global
6,BasicMotions,LOF OS,40,40,34,0.148605,0.050737,1,10,25,M-CELS Global
7,NATOPS,LOF OS,100,100,93,0.243264,0.228977,43,7,47,M-CELS Global
8,BasicMotions,Proximity (L2),40,40,34,66.704123,52.356071,13,0,21,M-CELS Global
9,NATOPS,Proximity (L2),100,100,93,9.634837,7.016293,29,0,64,M-CELS Global


Multi-SpaCE Sec. vs M-CELS Sec.


,dataset,metric_display,n_common_instances,n_common_valid_nuns,n_metric_instances,method_a_mean,method_b_mean,instance_wins_a,instance_ties,instance_wins_b,dataset_winner
0,BasicMotions,(Sparsity + Subsequences [%]) / 2,40,40,28,0.074107,0.055417,8,1,19,M-CELS Sec.
1,NATOPS,(Sparsity + Subsequences [%]) / 2,100,100,97,0.073357,0.028730,17,4,76,M-CELS Sec.
2,BasicMotions,AE OS,40,40,28,0.479878,0.492450,17,0,11,Multi-SpaCE Sec.
3,NATOPS,AE OS,100,100,97,0.629901,0.766685,70,0,27,Multi-SpaCE Sec.
4,BasicMotions,IF OS,40,40,28,0.418189,0.420300,13,1,14,Multi-SpaCE Sec.
5,NATOPS,IF OS,100,100,97,0.313051,0.320455,45,3,49,Multi-SpaCE Sec.
6,BasicMotions,LOF OS,40,40,28,0.230368,0.235559,8,10,12,Multi-SpaCE Sec.
7,NATOPS,LOF OS,100,100,97,0.239413,0.257429,50,10,41,Multi-SpaCE Sec.
8,BasicMotions,Proximity (L2),40,40,28,65.415469,59.073988,10,0,18,M-CELS Sec.
9,NATOPS,Proximity (L2),100,100,97,9.720892,8.071375,36,0,61,M-CELS Sec.


Multi-SpaCE Sec. vs M-CELS Sec. TF


,dataset,metric_display,n_common_instances,n_common_valid_nuns,n_metric_instances,method_a_mean,method_b_mean,instance_wins_a,instance_ties,instance_wins_b,dataset_winner
0,BasicMotions,(Sparsity + Subsequences [%]) / 2,40,40,0,NaN,NaN,0,0,0,n/a
1,NATOPS,(Sparsity + Subsequences [%]) / 2,100,100,0,NaN,NaN,0,0,0,n/a
2,BasicMotions,AE OS,40,40,0,NaN,NaN,0,0,0,n/a
3,NATOPS,AE OS,100,100,0,NaN,NaN,0,0,0,n/a
4,BasicMotions,IF OS,40,40,0,NaN,NaN,0,0,0,n/a
5,NATOPS,IF OS,100,100,0,NaN,NaN,0,0,0,n/a
6,BasicMotions,LOF OS,40,40,0,NaN,NaN,0,0,0,n/a
7,NATOPS,LOF OS,100,100,0,NaN,NaN,0,0,0,n/a
8,BasicMotions,Proximity (L2),40,40,0,NaN,NaN,0,0,0,n/a
9,NATOPS,Proximity (L2),100,100,0,NaN,NaN,0,0,0,n/a


Multi-SpaCE Sec. vs Multi-SpaCE Global


,dataset,metric_display,n_common_instances,n_common_valid_nuns,n_metric_instances,method_a_mean,method_b_mean,instance_wins_a,instance_ties,instance_wins_b,dataset_winner
0,BasicMotions,(Sparsity + Subsequences [%]) / 2,40,40,40,0.079708,0.101417,15,14,11,Multi-SpaCE Sec.
1,NATOPS,(Sparsity + Subsequences [%]) / 2,100,100,100,0.074677,0.062471,19,52,30,Multi-SpaCE Global
2,BasicMotions,AE OS,40,40,40,0.466997,0.341911,3,13,24,Multi-SpaCE Global
3,NATOPS,AE OS,100,100,100,0.635876,0.617711,23,51,26,Multi-SpaCE Global
4,BasicMotions,IF OS,40,40,40,0.396873,0.274597,1,13,26,Multi-SpaCE Global
5,NATOPS,IF OS,100,100,100,0.316801,0.287854,13,51,36,Multi-SpaCE Global
6,BasicMotions,LOF OS,40,40,40,0.189745,0.042183,0,13,27,Multi-SpaCE Global
7,NATOPS,LOF OS,100,100,100,0.245284,0.205698,20,51,29,Multi-SpaCE Global
8,BasicMotions,Proximity (L2),40,40,40,71.569553,67.666491,13,13,14,Multi-SpaCE Global
9,NATOPS,Proximity (L2),100,100,100,9.878399,8.221725,11,51,38,Multi-SpaCE Global
